#### simple self attention without trainable weights

In [1]:
import torch
inputs = torch.tensor(
    [[0.43, 0.15, 0.89], # Your     (x^1)
    [0.55, 0.87, 0.66], # journey  (x^2)
    [0.57, 0.85, 0.64], # starts   (x^3)
    [0.22, 0.58, 0.33], # with     
    [0.77, 0.25, 0.10], # one      
    [0.05, 0.80, 0.55]] # step     
)

In [15]:
query = inputs[1]
attn_scores_2 = torch.empty(inputs.shape[0])
for i, x_i in enumerate(inputs):
    attn_scores_2[i] = torch.dot(x_i,query)
print(attn_scores_2)


tensor([0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865])


In [16]:
attn_weights_2_tmp = attn_scores_2 / attn_scores_2.sum() # normalisation
print("Attention weights:", attn_weights_2_tmp)
print("Sum:", attn_weights_2_tmp.sum())

Attention weights: tensor([0.1455, 0.2278, 0.2249, 0.1285, 0.1077, 0.1656])
Sum: tensor(1.0000)


In [28]:
def softmax_naive(x):
    return torch.exp(x) / torch.exp(x).sum(dim=0)
softmax_naive(attn_scores_2)

tensor([0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581])

In [18]:
torch.softmax(attn_scores_2, dim=0)

tensor([0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581])

In [19]:
0.1385+0.2379+0.2333+ 0.1240+0.1082+0.1581

1.0

In [20]:
attn_scores = torch.empty(6,6)
for i , x_i in enumerate(inputs):
    for j , x_j in enumerate(inputs):
        attn_scores[i,j] = torch.dot(x_i,x_j)
print(attn_scores)

tensor([[0.9995, 0.9544, 0.9422, 0.4753, 0.4576, 0.6310],
        [0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865],
        [0.9422, 1.4754, 1.4570, 0.8296, 0.7154, 1.0605],
        [0.4753, 0.8434, 0.8296, 0.4937, 0.3474, 0.6565],
        [0.4576, 0.7070, 0.7154, 0.3474, 0.6654, 0.2935],
        [0.6310, 1.0865, 1.0605, 0.6565, 0.2935, 0.9450]])


In [ ]:
attn_scores = inputs @ inputs.T
attn_weights = torch.softmax(attn_scores , dim=1)
print(attn_weights) # 6x6

tensor([[0.2098, 0.2006, 0.1981, 0.1242, 0.1220, 0.1452],
        [0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581],
        [0.1390, 0.2369, 0.2326, 0.1242, 0.1108, 0.1565],
        [0.1435, 0.2074, 0.2046, 0.1462, 0.1263, 0.1720],
        [0.1526, 0.1958, 0.1975, 0.1367, 0.1879, 0.1295],
        [0.1385, 0.2184, 0.2128, 0.1420, 0.0988, 0.1896]])


In [26]:
torch.sum(torch.tensor([0.2098, 0.2006, 0.1981, 0.1242, 0.1220, 0.1452]))

tensor(0.9999)

In [ ]:
all_context_vecs = attn_weights @ inputs # 6x6 6x3 = 6 x 3
all_context_vecs

tensor([[0.4421, 0.5931, 0.5790],
        [0.4419, 0.6515, 0.5683],
        [0.4431, 0.6496, 0.5671],
        [0.4304, 0.6298, 0.5510],
        [0.4671, 0.5910, 0.5266],
        [0.4177, 0.6503, 0.5645]])

#### Implenting **Self Attention** with trainable weights

In [33]:
inputs

tensor([[0.4300, 0.1500, 0.8900],
        [0.5500, 0.8700, 0.6600],
        [0.5700, 0.8500, 0.6400],
        [0.2200, 0.5800, 0.3300],
        [0.7700, 0.2500, 0.1000],
        [0.0500, 0.8000, 0.5500]])

In [39]:
x_2 = inputs[1]
d_in = inputs.shape[1]
d_in
d_out = 2 # for showing purpose we are setting our own size , generally in and out are same size

torch.nn.Parameter is a special type of tensor in PyTorch that is used to represent learnable parameters of a neural network, such as weights and biases.

Not every tensor in a model should be trainable.
nn.Parameter explicitly tells PyTorch: this tensor should be optimized.

In [38]:
torch.manual_seed(123)
W_query = torch.nn.Parameter(torch.rand(d_in, d_out))  # .parameter makes the weights trainable makes it
W_key = torch.nn.Parameter(torch.rand(d_in, d_out))
W_value = torch.nn.Parameter(torch.rand(d_in, d_out))
W_query

Parameter containing:
tensor([[0.2961, 0.5166],
        [0.2517, 0.6886],
        [0.0740, 0.8665]], requires_grad=True)

In [41]:
query_2 = x_2 @ W_query #  1x3 3x2 = 1x2
query_2

tensor([0.4306, 1.4551], grad_fn=<SqueezeBackward4>)

In [55]:
keys = inputs @ W_key # 6x3 3x2
values = inputs @ W_value
keys.shape

torch.Size([6, 2])

In [44]:
keys

tensor([[0.3669, 0.7646],
        [0.4433, 1.1419],
        [0.4361, 1.1156],
        [0.2408, 0.6706],
        [0.1827, 0.3292],
        [0.3275, 0.9642]], grad_fn=<MmBackward0>)

In [43]:
keys_2 = keys[1]
attn_scores_2 = torch.dot(query_2,keys_2)
attn_scores_2

tensor(1.8524, grad_fn=<DotBackward0>)

In [47]:
attn_scores_2 = query_2 @ keys.T
attn_scores_2 # you can see the 1.8524 here too 

tensor([1.2705, 1.8524, 1.8111, 1.0795, 0.5577, 1.5440],
       grad_fn=<SqueezeBackward4>)

In [49]:
d_k = keys.shape[1]
attn_weights_2 = torch.softmax(attn_scores_2 / d_k**0.5 , dim = -1) # dim = -1 last dimension that is normalisation along columns or 
# you can say your weights of each word etc . good convention to follow 

In [51]:
print(attn_weights_2)
print(attn_weights_2.sum())

tensor([0.1500, 0.2264, 0.2199, 0.1311, 0.0906, 0.1820],
       grad_fn=<SoftmaxBackward0>)
tensor(1., grad_fn=<SumBackward0>)


In [57]:
context_vec_2 = attn_weights_2 @ values
context_vec_2


tensor([0.3061, 0.8210], grad_fn=<SqueezeBackward4>)

##### Implementing a compact self attention

In [59]:
"""
def __init__(self, d_in, d_out): — Constructor that takes:

d_in = input embedding dimension (e.g., 3 in your notebook)
d_out = output dimension for queries/keys/values (e.g., 2)
super().__init__() — Calls the parent nn.Module's constructor. 
                    This initializes the module properly so PyTorch can manage
                    it. Essential line—always include it when inheriting from nn.Module.
"""
import torch.nn as nn

class SelfAttention_v1(nn.Module):
    def __init__(self, d_in , d_out):
        super().__init__()
        self.d_out = d_out
        self.W_query = nn.Parameter(torch.rand(d_in , d_out))
        self.W_key = nn.Parameter(torch.rand(d_in , d_out))
        self.W_value = nn.Parameter(torch.rand(d_in , d_out))

    def forward(self , x): # say if our x is 6x3 and weights be 3x2
        keys = x @ self.W_key       # 6x2 
        queries = x @ self.W_query   # 6x2 
        values = x @ self.W_value        # 6x2 
        attn_scores = queries @ keys.T # omega  6x2 2x6
        attn_weights = torch.softmax(
            attn_scores / keys.shape[-1]**0.5 , dim = -1
        )
        context_vec = attn_weights @ values # 6x6 6x2
        return context_vec

torch.manual_seed(123)
sa_v1 = SelfAttention_v1(d_in,d_out)
sa_v1(inputs)

tensor([[0.2996, 0.8053],
        [0.3061, 0.8210],
        [0.3058, 0.8203],
        [0.2948, 0.7939],
        [0.2927, 0.7891],
        [0.2990, 0.8040]], grad_fn=<MmBackward0>)

In [65]:
m = torch.nn.Linear(2,3)
print(m.bias)
print(m.weight)

Parameter containing:
tensor([0.3311, 0.6207, 0.4322], requires_grad=True)
Parameter containing:
tensor([[-0.4761, -0.2816],
        [ 0.0284, -0.1649],
        [-0.0777, -0.6894]], requires_grad=True)


In [ ]:
import torch.nn as nn

class SelfAttention_v2(nn.Module):
    def __init__(self, d_in , d_out , qkv_bias = False):
        super().__init__()
        self.d_out = d_out
        self.W_query = nn.Linear(d_in , d_out ,bias=qkv_bias)
        self.W_key = nn.Linear(d_in , d_out)
        self.W_value = nn.Linear(d_in , d_out)

    def forward(self , x): # say if our x is 6x3 and weights be 3x2
        keys = self.W_key(x)      # 6x2 
        queries = self.W_query(x)   # 6x2 
        values = self.W_value(x)        # 6x2 
        attn_scores = queries @ keys.T # omega  6x2 2x6
        attn_weights = torch.softmax(
            attn_scores / keys.shape[-1]**0.5 , dim = -1
        )
        context_vec = attn_weights @ values # 6x6 6x2
        return context_vec

torch.manual_seed(123)  # if you change the seed number your outputs may vary from v1 and v2 cause of changing weights numbers
sa_v1 = SelfAttention_v1(d_in,d_out)
sa_v1(inputs)

tensor([[0.2996, 0.8053],
        [0.3061, 0.8210],
        [0.3058, 0.8203],
        [0.2948, 0.7939],
        [0.2927, 0.7891],
        [0.2990, 0.8040]], grad_fn=<MmBackward0>)

##### Applying casual attention mask